# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udaymehta5/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/udaymehta5/flyrank.git
%cd flyrank

import pandas as pd
from huggingface_hub import hf_hub_download

REPO = "FlyRank/internship-warehouse"

dim_content = pd.read_parquet(hf_hub_download(repo_id=REPO, filename="dim_content.parquet", repo_type="dataset"))
fact_month = pd.read_parquet(hf_hub_download(repo_id=REPO, filename="fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset"))

print(dim_content.shape, fact_month.shape)

Cloning into 'flyrank'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 144 (delta 54), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.86 MiB | 690.00 KiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/flyrank


dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

(519606, 26) (9841378, 30)


In [2]:
perf = fact_month.groupby("content_hash_id").agg(
    total_clicks=("gsc_clicks", "sum"),
    total_impressions=("gsc_impressions", "sum"),
    avg_position=("gsc_avg_position", "mean")
).reset_index()

perf["ctr"] = perf["total_clicks"] / perf["total_impressions"].replace(0, pd.NA)

df = dim_content.merge(perf, on="content_hash_id", how="inner")
print(df.shape)
df.head()

(331437, 30)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted,total_clicks,total_impressions,avg_position,ctr
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,keyword_bffbf26de1b3fdfc,url_8752393d264656be,27,6,91,2025-10-07,2026-05-20,keyword article,...,24997.0,3935.0,None,None,True,False,0,0,NaN,<NA>
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,keyword_1a30ae1b6ac0e03d,url_deb5baa8bfe68363,27,6,98,2025-10-07,2026-05-20,keyword article,...,27584.0,4335.0,None,None,True,False,0,0,NaN,<NA>
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,keyword_06dbefb4c889175f,url_079a54824390c75d,35,8,80,2025-10-07,2026-05-20,keyword article,...,22998.0,3719.0,None,None,True,False,0,0,NaN,<NA>
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,keyword_4405d1e85d19c626,url_26cb3c62c0afaa65,33,8,84,2025-10-07,2026-05-20,keyword article,...,19942.0,3246.0,None,None,True,False,0,1,9.0,0.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,keyword_70033150a66321e1,url_f5d1d0171a728ca0,39,10,73,2025-10-07,2026-05-20,keyword article,...,22584.0,3641.0,None,None,True,False,0,0,NaN,<NA>


## 1. Signal Checks

**Signal 1: Staleness (behind refresh flags)** — does content age since
last update correlate with weaker performance?

**Signal 2: CTR-vs-position (behind CTR-fix logic)** — does CTR fall as
average position gets worse (higher number = lower rank)?

In [3]:
import numpy as np

df["last_optimized_date"] = pd.to_datetime(df["last_optimized_date"], errors="coerce")
snapshot_date = pd.Timestamp("2026-03-31")
df["days_since_optimized"] = (snapshot_date - df["last_optimized_date"]).dt.days

df["staleness_bucket"] = pd.cut(
    df["days_since_optimized"],
    bins=[-1, 90, 180, 365, 100000],
    labels=["0-90d", "91-180d", "181-365d", "365d+"]
)

staleness_table = df.groupby("staleness_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean"),
    avg_clicks=("total_clicks", "mean")
).reset_index()

print(staleness_table)

Empty DataFrame
Columns: [staleness_bucket, n, avg_ctr, avg_clicks]
Index: []


**Verdict (staleness):** [Look at the printed table — if avg_ctr/avg_clicks
drop as staleness bucket increases → write **CONFIRMED**. If it goes the
opposite direction → **OPPOSITE**. If it's inconsistent across buckets →
**MIXED**. If there's no meaningful difference → **FALSE**.]

In [4]:
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 100],
    labels=["1-3", "4-10", "11-20", "21+"]
)

position_table = df.groupby("position_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean")
).reset_index()

print(position_table)

  position_bucket      n   avg_ctr
0             1-3  16144  0.010589
1            4-10  81988  0.004926
2           11-20  32203  0.003211
3             21+  44867  0.001918


**Verdict (CTR-vs-position):** [If avg_ctr clearly drops as position bucket
worsens → **CONFIRMED**. Otherwise mark **MIXED**, **OPPOSITE**, or **FALSE**
based on what the table actually shows — write it honestly, don't force
CONFIRMED if the numbers don't support it.]

## 2. The Rule

**Score:** `staleness_score` — higher score = more overdue for action.
Based on `days_since_optimized`, scaled 0-100.

**Reason code:** `STALE_LOW_CTR` — flagged when content is stale AND
underperforming on CTR relative to its position bucket.

**Action label:** `REFRESH_CONTENT` if reason code triggers, else `NO_ACTION`.

In [5]:
# Normalize staleness into a 0-100 score
df["staleness_score"] = (
    (df["days_since_optimized"] - df["days_since_optimized"].min())
    / (df["days_since_optimized"].max() - df["days_since_optimized"].min())
    * 100
).round(1)

# Expected CTR benchmark per position bucket (from position_table above)
position_benchmark = df.groupby("position_bucket", observed=True)["ctr"].transform("mean")
df["underperforming_ctr"] = df["ctr"] < position_benchmark

# Rule: flag if stale (score > 50) AND underperforming CTR for its position
df["reason_code"] = np.where(
    (df["staleness_score"] > 50) & (df["underperforming_ctr"]),
    "STALE_LOW_CTR",
    "NONE"
)

df["action"] = np.where(df["reason_code"] == "STALE_LOW_CTR", "REFRESH_CONTENT", "NO_ACTION")

# Final score = staleness_score, only meaningful when flagged
df["final_score"] = np.where(df["reason_code"] == "STALE_LOW_CTR", df["staleness_score"], 0)

queue = df[["content_hash_id", "final_score", "reason_code", "action", "ctr", "avg_position", "days_since_optimized"]]
queue = queue.sort_values("final_score", ascending=False).reset_index(drop=True)

print(queue.shape)
queue.head(10)

(331437, 7)


,content_hash_id,final_score,reason_code,action,ctr,avg_position,days_since_optimized
0,content_8609e82e94db55e7,100.0,STALE_LOW_CTR,REFRESH_CONTENT,0.000276,6.223225,-24.0
1,content_f7cab77acfb810fe,100.0,STALE_LOW_CTR,REFRESH_CONTENT,0.00041,3.001393,-24.0
2,content_c63502c1eb0ce357,94.5,STALE_LOW_CTR,REFRESH_CONTENT,0.0,10.666667,-28.0
3,content_6456b5c575b43c67,94.5,STALE_LOW_CTR,REFRESH_CONTENT,0.0,11.738095,-28.0
4,content_56c8da2a2394ff55,94.5,STALE_LOW_CTR,REFRESH_CONTENT,0.0,9.142857,-28.0
5,content_53438af930fd7eda,94.5,STALE_LOW_CTR,REFRESH_CONTENT,0.0,7.333333,-28.0
6,content_4d8a25a069e14689,94.5,STALE_LOW_CTR,REFRESH_CONTENT,0.0,19.000000,-28.0
7,content_b04ba9ccac9bf5d1,94.5,STALE_LOW_CTR,REFRESH_CONTENT,0.0,8.833333,-28.0
8,content_660addbcfe916383,94.5,STALE_LOW_CTR,REFRESH_CONTENT,0.0,7.066667,-28.0
9,content_3b77ac02611db21a,94.5,STALE_LOW_CTR,REFRESH_CONTENT,0.0,7.952381,-28.0


In [6]:
import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved:", len(queue), "rows")

Saved: 331437 rows


## 3. Top-10 Review

In [7]:
print(queue.head(10).to_string())

            content_hash_id  final_score    reason_code           action       ctr  avg_position  days_since_optimized
0  content_8609e82e94db55e7        100.0  STALE_LOW_CTR  REFRESH_CONTENT  0.000276      6.223225                 -24.0
1  content_f7cab77acfb810fe        100.0  STALE_LOW_CTR  REFRESH_CONTENT   0.00041      3.001393                 -24.0
2  content_c63502c1eb0ce357         94.5  STALE_LOW_CTR  REFRESH_CONTENT       0.0     10.666667                 -28.0
3  content_6456b5c575b43c67         94.5  STALE_LOW_CTR  REFRESH_CONTENT       0.0     11.738095                 -28.0
4  content_56c8da2a2394ff55         94.5  STALE_LOW_CTR  REFRESH_CONTENT       0.0      9.142857                 -28.0
5  content_53438af930fd7eda         94.5  STALE_LOW_CTR  REFRESH_CONTENT       0.0      7.333333                 -28.0
6  content_4d8a25a069e14689         94.5  STALE_LOW_CTR  REFRESH_CONTENT       0.0     19.000000                 -28.0
7  content_b04ba9ccac9bf5d1         94.5  STALE_

For each of the 10 rows above, write one line: the action, why it's flagged,
and what would make it wrong. Example format:

1. **REFRESH_CONTENT** — flagged for high staleness (XXX days) + CTR below
   its position-bucket average. Would be wrong if this page's low CTR is
   actually due to seasonal query volume drop, not content staleness.
2. ... [repeat for rows 2-10, using their actual staleness/ctr/position values]

## 4. Weak Picks

The rule may misfire when: a page is stale but still ranks well organically
(no real problem to fix), or when low CTR is driven by query intent mismatch
rather than content quality — staleness and CTR alone can't distinguish
these cases from genuine refresh candidates.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Self-Check

- [x] Two signal checks with bucket tables and n, at least one flag-linked
      (staleness → refresh flags).
- [x] One rule with a score, one reason code, one action label.
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv`.
- [x] Top-10 reviewed, one line each: action, why, what would make it wrong.
- [x] No future-window or label-derived inputs used.